# BO Forge three-variable LogEI campaign

This short notebook demonstrates the `CampaignSession` workflow for a 3D continuous campaign.

The key diagnostic point is that 3D and higher-dimensional campaigns cannot be fully understood from one static scatter plot. BO Forge therefore uses progress plus variable-coverage diagnostics for 3D+ spaces.

## 1. Setup

The public campaign log keeps variables in original user units. Internal normalisation is used only for modelling and diagnostics.

In [ ]:
from pathlib import Path
import os
import shutil
import sys

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

mpl_cache = PROJECT_ROOT / ".matplotlib-cache"
mpl_cache.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from bo_forge.session import CampaignSession

In [ ]:
config_path = PROJECT_ROOT / "configs" / "03_simple_3d_maximise_logei.yaml"
seed_log_path = PROJECT_ROOT / "examples" / "03_simple_3d_maximise_logei_campaign_log.csv"
working_log_path = PROJECT_ROOT / "examples" / "03_simple_3d_maximise_logei_working_log.csv"
latest_suggestion_path = PROJECT_ROOT / "examples" / "03_simple_3d_maximise_logei_latest_suggestions.csv"
report_path = PROJECT_ROOT / "reports" / "03_simple_3d_campaign_report.txt"
progress_path = PROJECT_ROOT / "reports" / "03_simple_3d_progress.pdf"
diagnostics_path = PROJECT_ROOT / "reports" / "03_simple_3d_diagnostics.pdf"
TARGET_OBSERVED_ROWS = 15

shutil.copyfile(seed_log_path, working_log_path)
campaign = CampaignSession.from_files(config_path=config_path, log_path=working_log_path)

## 2. Synthetic objective

The synthetic activity function has smooth curvature, a clear optimum, and a mild interaction between variables.

In [ ]:
def simulate_activity(row) -> float:
    x1 = float(row["precursor_ratio"])
    x2 = (float(row["annealing_temperature"]) - 300.0) / 500.0
    x3 = (float(row["electrolyte_concentration"]) - 0.1) / 1.9

    peak = np.exp(
        -0.5
        * (
            ((x1 - 0.62) / 0.16) ** 2
            + ((x2 - 0.55) / 0.20) ** 2
            + ((x3 - 0.35) / 0.18) ** 2
        )
    )
    interaction = 0.15 * np.sin(4.0 * x1 + 2.0 * x3)
    return float(1.0 + peak + interaction)

## 3. Inspect campaign state

In [ ]:
campaign.validate()
display(campaign.summary())
display(campaign.next_action())
campaign.df

## 4. Suggest and record one experiment

In [ ]:
suggestion = campaign.suggest_next(batch_size=1)
suggestion.to_csv(latest_suggestion_path, index=False)
campaign.append_suggestions(suggestion)
display(campaign.pending_suggestions())

In [ ]:
row = suggestion.iloc[0]
activity = simulate_activity(row)
campaign.mark_observed(str(row["row_id"]), activity)
campaign.reload()
display(campaign.summary())
campaign.df.tail(3)

## 5. Run a few more sequential experiments

Continue the same campaign loop one experiment at a time. Starting from the seed log plus the first recorded suggestion, these extra simulated runs finish the Sobol initial design and then trigger one LogEI model-based suggestion.

In [ ]:
def run_one_simulated_experiment() -> pd.DataFrame:
    suggestion = campaign.suggest_next(batch_size=1)
    suggestion.to_csv(latest_suggestion_path, index=False)
    campaign.append_suggestions(suggestion)

    row = suggestion.iloc[0]
    activity = simulate_activity(row)
    campaign.mark_observed(str(row["row_id"]), activity)
    campaign.reload()

    recorded = campaign.df.loc[campaign.df["row_id"] == row["row_id"]].copy()
    return recorded

In [ ]:
additional_runs = []
while len(campaign.observed_data()) < TARGET_OBSERVED_ROWS:
    additional_runs.append(run_one_simulated_experiment())

campaign.reload()
assert len(campaign.observed_data()) == TARGET_OBSERVED_ROWS

additional_results = pd.concat(additional_runs, ignore_index=True)
display(additional_results[[
    "row_id",
    "iteration",
    "source",
    "precursor_ratio",
    "annealing_temperature",
    "electrolyte_concentration",
    "activity",
]])
display(campaign.summary())
display(campaign.next_action())

## 6. Report and diagnostics

In [ ]:
report = campaign.report()
campaign.export_report(report_path)
display(report["summary"])
display(report["best_observation"])

For 3D+ campaigns, `plot_diagnostics()` shows progress and variable coverage. It does not show full interaction structure across all variables.

In [ ]:
campaign.plot_progress(save_path=progress_path);

In [ ]:
campaign.plot_diagnostics(save_path=diagnostics_path);